In [8]:
import pyspark
import re
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder.appName('BigD_OurD').getOrCreate()
sc = spark.sparkContext

In [9]:
########################
# Act 3: Partitioning  #
########################

# Initialization of the whole RDD
def csvFilter(line):
    out = []
    if "\"" not in line:
        return line.split(",")
    else:
        start = 0
        indices = [match.start() for match in re.finditer(r",", line)]
        while len(indices) > 10:
            indices.pop(1)
        indices.append(-1)
        for end in indices:
             out.append(line[start:end])
             start = end+1
        return out

game_rdd = sc.textFile("vgsales.csv").map(csvFilter)
header = game_rdd.first()
game_rdd = game_rdd.filter(lambda x: x != header)

In [10]:
###################################
# Genre to Global Sale Comparison #
#    by Keiron Mirandilla         #
###################################

# Keep genre(5th) and global sales(last) as a tuple w/ Hash Partitioning
# Get sum of global sales and sort by highest
genre_sales = game_rdd.map(lambda x: (x[4], float(x[-1]))).partitionBy(4).persist()
summed_sales = genre_sales.reduceByKey(lambda a,b: a+b).sortBy(lambda x: x[1], ascending=False)

top_genres = summed_sales.take(3)

In [11]:
print("Top 3 Genres by Global Sales:")
for i in range(3):
    print(f"  {i+1}. {top_genres[i][0]} with {top_genres[i][1]:.2f} million sales.")

Top 3 Genres by Global Sales:
  1. Action with 1748.75 million sales.
  2. Sports with 1330.73 million sales.
  3. Shooter with 1035.85 million sales.


In [12]:
########################
# Top Gaming Platforms #
#    by Edgar          #
########################
from pyspark.sql.functions import sum as _sum, desc


platform_total_sales = df_cleaned.groupBy('Platform') \
    .agg(_sum('Global_Sales').alias('Total_Global_Sales'))

# rp
platform_total_sales = platform_total_sales.repartitionByRange(4, 'Total_Global_Sales')

# sort by most sales most to least top 10
platform_total_sales = platform_total_sales.orderBy(desc('Total_Global_Sales'))

platform_total_sales.show(10)

top_platform = platform_total_sales.first()
print(f"The platform with the most global sales is {top_platform['Platform']} with {top_platform['Total_Global_Sales']:.2f} million units.")
print(f"Number of partitions: {platform_total_sales.rdd.getNumPartitions()}")

NameError: name 'df_cleaned' is not defined

In [13]:
##############################
# Act 4: Data Preprocessing  #
#        and DataFrames      #
##############################

# Initialization and pre-processing of dataframe and schema
headers = [
    ("Show ID", StringType(), False),
    ("Type", StringType(), False),
    ("Title", StringType(), False),
    ("Director", StringType(), True),
    ("Cast", StringType(), True),
    ("Country", StringType(), True),
    ("Date Added", DateType(), True),
    ("Release Year", IntegerType(), False),
    ("Rating", StringType(), True),
    ("Duration", StringType(), True),
    ("Listed In", StringType(), False),
    ("Description", StringType(), False)
]

netflix_schema = StructType([
    StructField(*field) for field in headers
])
netflix_df = spark.read.csv("netflix_titles.csv", header=True, schema=netflix_schema)

# Get headers with missing values and
netflix_df = netflix_df.fillna({key[0]: "N/A" for key in headers if key[2]})

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/content/netflix_titles.csv. SQLSTATE: 42K03

In [ ]:
netflix_df.select(col("Title"),col("Director")).show()

+--------------------+--------------------+
|               Title|            Director|
+--------------------+--------------------+
|Dick Johnson Is Dead|     Kirsten Johnson|
|       Blood & Water|                 N/A|
|           Ganglands|     Julien Leclercq|
|Jailbirds New Orl...|                 N/A|
|        Kota Factory|                 N/A|
|       Midnight Mass|       Mike Flanagan|
|My Little Pony: A...|Robert Cullen, Jo...|
|             Sankofa|        Haile Gerima|
|The Great British...|     Andy Devonshire|
|        The Starling|      Theodore Melfi|
|Vendetta: Truth, ...|                 N/A|
|    Bangkok Breaking|   Kongkiat Komesiri|
|        Je Suis Karl| Christian Schwochow|
|Confessions of an...|       Bruno Garotti|
|Crime Stories: In...|                 N/A|
|   Dear White People|                 N/A|
|Europe's Most Dan...|Pedro de Echave G...|
|     Falsa identidad|                 N/A|
|           Intrusion|          Adam Salky|
|              Jaguar|          

In [15]:
# Nash A. Mapula
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql import functions as F

# CREATE OWN SCHEMA
vgsales_schema = StructType([
    StructField("Rank", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("Platform", StringType(), True),
    StructField("Year", IntegerType(), True),
    StructField("Genre", StringType(), True),
    StructField("Publisher", StringType(), True),
    StructField("NA_Sales", DoubleType(), True),
    StructField("EU_Sales", DoubleType(), True),
    StructField("JP_Sales", DoubleType(), True),
    StructField("Other_Sales", DoubleType(), True),
    StructField("Global_Sales", DoubleType(), True)
])

# LOAD & PRE-PROCESS (Clean missing data and remove duplicates)
df = spark.read.csv("/content/vgsales.csv", header=True, schema=vgsales_schema)

# Cleaning: Remove rows where Year or Name is null, and remove any duplicates
df_cleaned = df.filter(F.col("Year").isNotNull()).dropDuplicates()

# DETERMINE INSIGHTS (Applying Range Partition logic)
# partition by Year range as specific strategy
partitioned_df = df_cleaned.repartitionByRange(5, "Year")

# Insight: Total Action Genre Sales per Year
my_analysis = partitioned_df.filter(F.col("Genre") == "Action") \
                            .groupBy("Year") \
                            .agg(F.sum("Global_Sales").alias("Total_Action_Sales")) \
                            .sort("Year")

my_analysis.show(10)

# WRITE TO PARQUET
my_analysis.write.mode("overwrite").parquet("nash_action_sales_analysis.parquet")
print("Analysis successfully saved to Parquet!")

+----+------------------+
|Year|Total_Action_Sales|
+----+------------------+
|1980|              0.34|
|1981|14.839999999999996|
|1982|              6.52|
|1983|              2.86|
|1984|              1.85|
|1985|              3.52|
|1986|13.740000000000002|
|1987|              1.12|
|1988|              1.75|
|1989|              4.64|
+----+------------------+
only showing top 10 rows
Analysis successfully saved to Parquet!
